# Triton Kernel 主线 · 第 6/10 课：均值方差与 Mask 二次传播

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现 population variance，并解释 padding 在中心化后为何会重新变成非零。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：均值和方差是两次 reduction；masked load 的 0 对 sum 安全，但减 mean 后 padding lane 变成 `-mean`。

## 核心心智模型

### 1. 它是什么，解决什么问题

均值和方差是两次 reduction；masked load 的 0 对 sum 安全，但减 mean 后 padding lane 变成 `-mean`。

### 2. 它如何工作

先算 mean，再对平方差显式 `where(mask,...,0)`，最后除真实 N 而非 BLOCK。

### 3. 正确性条件与常见误区

必须区分 population variance 与 unbiased sample variance；N=1 时后者无定义。

### 4. 性能与工程取舍

两遍统计简单但有额外运算；Welford 更稳定、combine 更复杂。

## 具体演示

N=3、BLOCK=4，padding 值 0；若 mean=2，padding 的平方差是 4，未重新 mask 会污染方差。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐二次 mask。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def mean_var_kernel(x, means, variances, N: tl.constexpr,
                    stride_m: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < N
    values = tl.load(x + row * stride_m + cols, mask=mask, other=0.0)
    mean = tl.sum(values, axis=0) / N
    centered_sq = tl.where(______, (values - mean) * (values - mean), 0.0)  # TODO
    var = tl.sum(centered_sq, axis=0) / N
    tl.store(means + row, mean)
    tl.store(variances + row, var)

def mean_var(x):
    assert x.ndim == 2 and x.stride(1) == 1
    M, N = x.shape
    means = torch.empty(M, device=x.device, dtype=x.dtype)
    variances = torch.empty_like(means)
    mean_var_kernel[(M,)](x, means, variances, N, x.stride(0),
                          BLOCK=triton.next_power_of_2(N))
    return means, variances

for shape in ((3, 17), (4, 128)):
    x = torch.randn(shape, device="cuda")
    m, v = mean_var(x)
    torch.testing.assert_close(m, x.mean(1), atol=1e-4, rtol=1e-4)
    torch.testing.assert_close(v, x.var(1, unbiased=False), atol=2e-4, rtol=2e-4)


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“均值方差与 Mask 二次传播”的工作机制。

**你的答案：**


### Q2

只在 load 时 mask 为什么仍会污染方差？

**你的答案：**


### Q3

大均值小方差时如何改进数值稳定性？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。